# 步驟六：模型評估（Evaluation）執行指南

在完成隨機森林、XGBoost、DNN 三個模型的訓練後，評估階段的核心目標是：驗證模型在未見過數據（驗證集/測試集）上的泛化能力，並挑選出最能平衡「公司授信收益」與「違約損失」的模型與決策門檻。

以下是具體的執行步驟與思考框架：

## 步驟 1：預測「機率值」而非直接預測「類別」

許多人在評估時會直接調用模型的預測類別（0 或 1），但在信用風險（先買後付）預測中，預測機率（Probability Scores）才是靈魂。

原因：預設的分類門檻 $0.5$ 通常不適用於極度不平衡且「偽陰性代價高」的違約場景。

做法：對驗證集/測試集進行預測時，請確保輸出的結果是「用戶違約的機率值」（介於 $0$ 到 $1$ 之間）。

## 步驟 2：繪製「雙曲線」進行模型整體預測能力的對比

在調整任何決策門檻之前，必須先評估模型「區分好壞客戶」的本質能力。請繪製以下兩張圖表：

1. ROC 曲線與 AUC (Area Under ROC)

原理：以 $False\ Positive\ Rate$（橫軸）與 $True\ Positive\ Rate\ (Recall)$（縱軸）繪製的曲線。

評估點：AUC 越接近 $1.0$ 代表模型對「準時付款」與「違約」的排序能力越好。

注意：ROC 曲線在類別不平衡（如違約率僅 3% ~ 5%）時容易顯得過於樂觀。

## 步驟 3：計算預設門檻 ($0.5$) 下的基本分類指標

雖然我們會優化門檻，但仍需先建立一個基準（Baseline）對照組。請計算三個模型在預期門檻 $0.5$ 下的以下指標：

混淆矩陣 (Confusion Matrix)：

統計出：真陽性 (TP)、真陰性 (TN)、偽陽性 (FP)、偽陰性 (FN)。

請特別標出 FN（漏報違約） 的個數，這就是你步驟一提到「代價最高」的部分。

準確率 (Accuracy)：


$$Accuracy = \frac{TP + TN}{TP + TN + FP + FN}$$

警告：僅供參考。若違約率僅 5%，盲猜所有人不違約便能得到 95% 準確率，這完全沒有商業價值。

精準度 (Precision)：


$$Precision = \frac{TP}{TP + FP}$$

商業意義：在模型預測「會違約」的人群中，實際上真的違約的比例。代表風控審核的精準度（避免誤殺好人）。

召回率 (Recall / Sensitivity)：


$$Recall = \frac{TP}{TP + FN}$$

商業意義：在所有「實際违約」的客戶中，模型成功抓出了多少人。這是你最需要提高的指標（FN 越低，Recall 越高）。

$F_\beta$-Score（推薦使用 $F_2$-Score）：


$$F_\beta = (1 + \beta^2) \frac{Precision \times Recall}{\beta^2 Precision + Recall}$$

商業意義：傳統的 $F_1$-Score 平衡了 Precision 與 Recall。但在你的場景中，Recall 的重要性大於 Precision。

設定：建議將 $\beta$ 設為 $2$（即 $F_2$-Score），這會給予 Recall 兩倍於 Precision 的權重，非常符合金融風控的本質。